# Stock Risk & Return Analysis

## Project Overview
This project analyzes the historical risk and return characteristics of five major technology companies:

- Apple
- Amazon
- Facebook
- Google
- Microsoft

The analysis uses Python to clean, explore, and calculate key financial metrics, then compares the results with the Excel and Power BI analysis.

### Financial Metrics
- Average Daily Return
- Annualized Return
- Annualized Volatility
- Sharpe Ratio
- Maximum Drawdown

> **Note:** The results describe historical performance in the supplied dataset and are not investment advice.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## 2. Load the Dataset

The project dataset contains historical stock-price observations and daily returns.

In [ ]:
# Update this path if you upload the SQL/CSV source to Google Colab.
file_path = "FAANG_Combined_Data_Separated.xlsx"

# Load the workbook and inspect available sheets
xls = pd.ExcelFile(file_path)
xls.sheet_names


In [ ]:
# Load the combined historical-data sheet
# Change the sheet name below if your workbook uses a different name.
df = pd.read_excel(file_path, sheet_name=0)

df.head()


## 3. Understand the Data

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 4. Data Cleaning

In [ ]:
# Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# Convert date column when present
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Sort observations
if {"company", "date"}.issubset(df.columns):
    df = df.sort_values(["company", "date"]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
display(df.head())


## 5. Exploratory Data Analysis

In [ ]:
# Summary statistics for numeric columns
display(df.describe(include="all"))


In [ ]:
# Number of observations by company
if "company" in df.columns:
    display(df["company"].value_counts())


## 6. Daily Returns

In [ ]:
# If a daily-return column already exists, use it.
# Otherwise calculate simple daily returns from adjusted/close price.

if "daily_return" not in df.columns:
    price_col = next(
        (c for c in ["adj_close", "adjusted_close", "close"] if c in df.columns),
        None
    )

    if price_col is None:
        raise ValueError("No daily_return or price column was found.")

    df["daily_return"] = (
        df.groupby("company")[price_col]
          .pct_change()
    )

display(df[["company", "date", "daily_return"]].head(10))


## 7. Average Daily Return

In [ ]:
avg_daily_return = (
    df.groupby("company")["daily_return"]
      .mean()
      .rename("Average Daily Return")
)

display((avg_daily_return * 100).round(3).sort_values(ascending=False))


## 8. Annualized Return

In [ ]:
# Annualized return based on the mean daily return:
annualized_return = (
    avg_daily_return * 252
).rename("Annualized Return")

display((annualized_return * 100).round(2).sort_values(ascending=False))


## 9. Annualized Volatility

In [ ]:
# Standard deviation of daily returns annualized using 252 trading days.
annualized_volatility = (
    df.groupby("company")["daily_return"].std() * np.sqrt(252)
).rename("Annualized Volatility")

display((annualized_volatility * 100).round(2).sort_values())


## 10. Sharpe Ratio

In [ ]:
# Risk-free rate = 0%, consistent with the project's Risk_Return_Analysis sheet.
risk_free_rate = 0.0

sharpe_ratio = (
    (annualized_return - risk_free_rate) / annualized_volatility
).rename("Sharpe Ratio")

display(sharpe_ratio.round(2).sort_values(ascending=False))


## 11. Maximum Drawdown

In [ ]:
# Calculate the maximum peak-to-trough decline from the adjusted/close price.
price_col = next(
    (c for c in ["adj_close", "adjusted_close", "close"] if c in df.columns),
    None
)

if price_col is None:
    raise ValueError("No price column was found for drawdown calculation.")

def maximum_drawdown(group):
    prices = group[price_col].dropna()
    running_peak = prices.cummax()
    drawdown = prices / running_peak - 1
    return drawdown.min()

max_drawdown = (
    df.groupby("company")
      .apply(maximum_drawdown, include_groups=False)
      .rename("Maximum Drawdown")
)

display((max_drawdown * 100).round(2).sort_values(ascending=False))


## 12. Final Company Comparison

In [ ]:
summary = pd.concat(
    [
        avg_daily_return,
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        max_drawdown
    ],
    axis=1
)

summary = summary.sort_values("Annualized Return", ascending=False)

display(
    summary.style.format({
        "Average Daily Return": "{:.3%}",
        "Annualized Return": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Sharpe Ratio": "{:.2f}",
        "Maximum Drawdown": "{:.2%}"
    })
)


## 13. Risk–Return Visualization

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    summary["Annualized Volatility"] * 100,
    summary["Annualized Return"] * 100,
    s=100
)

for company, row in summary.iterrows():
    plt.annotate(
        company,
        (
            row["Annualized Volatility"] * 100,
            row["Annualized Return"] * 100
        ),
        xytext=(6, 6),
        textcoords="offset points"
    )

plt.xlabel("Annualized Volatility (%)")
plt.ylabel("Annualized Return (%)")
plt.title("Risk–Return Comparison")
plt.grid(alpha=0.25)
plt.show()


## 14. Metric Comparison Charts

In [ ]:
# Annualized Return
summary["Annualized Return"].sort_values().mul(100).plot(
    kind="barh", figsize=(9, 5), title="Annualized Return by Company"
)
plt.xlabel("Annualized Return (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Annualized Volatility
summary["Annualized Volatility"].sort_values().mul(100).plot(
    kind="barh", figsize=(9, 5), title="Annualized Volatility by Company"
)
plt.xlabel("Annualized Volatility (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Sharpe Ratio
summary["Sharpe Ratio"].sort_values().plot(
    kind="barh", figsize=(9, 5), title="Sharpe Ratio by Company"
)
plt.xlabel("Sharpe Ratio")
plt.tight_layout()
plt.show()


In [ ]:
# Maximum Drawdown
summary["Maximum Drawdown"].sort_values().mul(100).plot(
    kind="barh", figsize=(9, 5), title="Maximum Drawdown by Company"
)
plt.xlabel("Maximum Drawdown (%)")
plt.tight_layout()
plt.show()


## 15. Key Financial Insights

In [ ]:
best_return = summary["Annualized Return"].idxmax()
lowest_volatility = summary["Annualized Volatility"].idxmin()
best_sharpe = summary["Sharpe Ratio"].idxmax()
smallest_drawdown = summary["Maximum Drawdown"].idxmax()
largest_drawdown = summary["Maximum Drawdown"].idxmin()

print(f"Highest annualized return: {best_return}")
print(f"Lowest annualized volatility: {lowest_volatility}")
print(f"Highest Sharpe ratio: {best_sharpe}")
print(f"Smallest maximum drawdown: {smallest_drawdown}")
print(f"Largest maximum drawdown: {largest_drawdown}")


## 16. Conclusion

The analysis compares historical return, volatility, risk-adjusted performance, and downside risk across Apple, Amazon, Facebook, Google, and Microsoft.

The results should be interpreted as **historical observations from the supplied dataset**, not as a recommendation to buy or sell any security.

The final findings are carried into the Power BI dashboard for visual analysis.
